In [21]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, ICRS, get_sun
import astropy.units as u
from astropy.wcs import WCS
from astropy.time import Time
import requests
from skyfield.api import load, wgs84

from gen2_starlink import satellitemodels

# Load ephemeris and timescale
ts = load.timescale()
eph = load('de440s.bsp')
sun = eph['Sun']
earth = eph['Earth']


def healpix_to_lonlat(healpix_id, nside=16):
    theta, phi = hp.pix2ang(nside, healpix_id, nest=False)
    lat = 90 - np.degrees(theta)
    lon = np.degrees(phi)
    lon = ((lon + 180) % 360) - 180
    return lon, lat

In [22]:
def horizontal_to_equatorial(latitude, longitude, elevation, azimuth_deg, elevation_obs_deg, julian_day):
    location = EarthLocation(lat=latitude*u.deg, lon=longitude*u.deg, height=elevation*u.m)
    t = Time(julian_day, format='jd')
    altaz = AltAz(az=azimuth_deg*u.deg, alt=elevation_obs_deg*u.deg, location=location, obstime=t)
    eq = altaz.transform_to(ICRS())
    return eq.ra.deg, eq.dec.deg

In [23]:
import astropy

import numpy as np
import astropy.time
import astropy.coordinates
import astropy.units as u


healpix_ids       = np.array([891, 963, 1025, 955, 831, 896])
observation_times = np.array([2460602.371181, 2460602.368403, 2460602.364931, 2460602.371875, 2460602.369792, 2460602.369097])
min_magnitudes    = np.array([7, 7, 6.97, 4.09, 5.36, 5.37])
avg_magnitudes    = np.array([7, 7, 6.97, 6.19, 5.64, 5.97])

lat, lon, elev = -29.038169452952143, 26.402995668269295, 1372
duration, fov = 60, 3.0

    # First print all the info
for hpid, obs, mn, av in zip(healpix_ids, observation_times, min_magnitudes, avg_magnitudes):
    azi, ele = healpix_to_lonlat(hpid)
    ra_c, dec_c = horizontal_to_equatorial(lat, lon, elev, azi, ele, obs)
        
    print(f"Healpix ID {hpid}:")
    print(f"  Observation Time (JD): {obs}")
    print(f"  Azimuth: {azi:.2f}°, Elevation: {ele:.2f}°")
    print(f"  RA: {ra_c:.6f}°, Dec: {dec_c:.6f}°")
    print()

Healpix ID 891:
  Observation Time (JD): 2460602.371181
  Azimuth: 154.69°, Elevation: 24.62°
  RA: 102.945969°, Dec: -67.009868°

Healpix ID 963:
  Observation Time (JD): 2460602.368403
  Azimuth: -163.12°, Elevation: 22.02°
  RA: 255.219540°, Dec: -73.218955°

Healpix ID 1025:
  Observation Time (JD): 2460602.364931
  Azimuth: -171.56°, Elevation: 19.47°
  RA: 225.496807°, Dec: -77.634596°

Healpix ID 955:
  Observation Time (JD): 2460602.371875
  Azimuth: 151.88°, Elevation: 22.02°
  RA: 107.031540°, Dec: -63.710333°

Healpix ID 831:
  Observation Time (JD): 2460602.369792
  Azimuth: 174.38°, Elevation: 27.28°
  RA: 116.572319°, Dec: -84.686497°

Healpix ID 896:
  Observation Time (JD): 2460602.369097
  Azimuth: -177.19°, Elevation: 24.62°
  RA: 216.227783°, Dec: -84.810601°



In [24]:
import astropy
import requests
import json
from astropy.coordinates import EarthLocation
import numpy as np
import astropy.time
import astropy.coordinates
import astropy.units as u

def sun_altazi(time, ob_location):
    obs_time = astropy.time.Time(time, format='jd')  
    aa_frame = astropy.coordinates.AltAz(obstime=obs_time, location=ob_location)
    sun_altaz = astropy.coordinates.get_sun(obs_time).transform_to(aa_frame)
    
    sun_alt = np.array(sun_altaz.alt.degree)  
    sun_az = np.array(sun_altaz.az.degree)
    
    print("Sun Altitudes:", sun_alt)
    print("Sun Azimuths:", sun_az)
    
    # If it's a single-element array, just return the scalar values
    if sun_alt.size == 1:
        return sun_az.item(), sun_alt.item()
    else:
        return sun_az, sun_alt




# All your Julian dates
observation_times = np.array([2460602.371181, 2460602.368403, 2460602.364931, 
                             2460602.371875, 2460602.369792, 2460602.369097])

#Start of function:
def parameterfunction(jd_time, ra_c, dec_c):

    all_parameters = []
    # Define Earth location with height
    south_af = astropy.coordinates.EarthLocation(lat=-29.038169452952143*u.deg, 
                                                lon=26.402995668269295*u.deg, 
                                                height=1372*u.m)
    
    
    # API base URL
    url = 'https://satchecker.cps.iau.org/fov/satellite-passes/'    
    print(jd_time)    
        # API parameters for this specific Julian date
    params = {
        'latitude': -29.038169452952143, 
        'longitude': 26.402995668269295, 
        'elevation': 1372, 
        'start_time_jd': jd_time,  # Use the current Julian date
        'duration': 60, 
        'ra': ra_c, 
        'dec': dec_c, 
        'fov_radius': 3, 
        'group_by': 'satellite'
    }
    try:
        # Make API request
        r = requests.get(url, params=params)
        data = r.json()
        print(jd_time) 
        # Check if we got valid data
        if 'data' in data and 'satellites' in data['data']:
            # Process each satellite and each position for this Julian date
            for satellite_name, satellite_data in data['data']['satellites'].items():
                print(f"\nProcessing satellite: {satellite_name}")
                
                for position in satellite_data['positions']:
                    # Extract the data you need
                    observation_time = position['julian_date']
                    satellite_height = position['range_km']  # in Km
                    azimuth = position['azimuth']  # in degrees
                    altitude = position['altitude']  # in degrees
                    
                    # Get sun position
                    sun_azimuth, sun_altitude = sun_altazi([observation_time], south_af)
                    
                    # Create parameter array
                    params_array = np.array([
                        satellite_height,
                        azimuth,
                        altitude,
                        sun_azimuth,
                        sun_altitude
                    ])

                    all_parameters.append({
                        'julian_date': jd_time,
                        'satellite': satellite_name,
                        'parameters': params_array,
                    })
                    
                    print(f"  Time: {observation_time:.6f}")
                    print(f"  Height: {satellite_height:.2f} km")
                    print(f"  Azimuth: {azimuth:.2f}°, Altitude: {altitude:.2f}°")
                    print(f"  Sun Azimuth: {sun_azimuth:.2f}°, Sun Altitude: {sun_altitude:.2f}°")
                    print(f"  Parameters array: {params_array}")
                    print(f"  Right Accension: {ra_c}")
                    print(f"  Declination: {dec_c}")
        else:
            print(f"No satellite data found for Julian date {jd_time}")
            
    except Exception as e:
        print(f"Error processing Julian date {jd_time}: {e}")
    return all_parameters

In [25]:
a = parameterfunction(2460602.371181, 102.945969, -67.009868)
b = parameterfunction(2460602.368403, 255.219540, -73.218955)
c = parameterfunction(2460602.364931, 225.496807, -77.634596)
d = parameterfunction(2460602.371875, 107.031540, -63.710333)
e = parameterfunction(2460602.369792, 116.572319, -84.686497)
F = parameterfunction(2460602.369097, 216.227783, -84.810601)

2460602.371181
2460602.371181

Processing satellite: GLOBALSTAR M056 (25945)
Sun Altitudes: [-47.91604432]
Sun Azimuths: [204.22193462]
  Time: 2460602.371181
  Height: 2849.99 km
  Azimuth: 153.26°, Altitude: 26.18°
  Sun Azimuth: 204.22°, Sun Altitude: -47.92°
  Parameters array: [2849.99368258  153.25932953   26.17698074  204.22193462  -47.91604432]
  Right Accension: 102.945969
  Declination: -67.009868
Sun Altitudes: [-47.91753445]
Sun Azimuths: [204.21623248]
  Time: 2460602.371193
  Height: 2854.31 km
  Azimuth: 153.20°, Altitude: 26.09°
  Sun Azimuth: 204.22°, Sun Altitude: -47.92°
  Parameters array: [2854.30719748  153.19643564   26.09156041  204.21623248  -47.91753445]
  Right Accension: 102.945969
  Declination: -67.009868
Sun Altitudes: [-47.91902558]
Sun Azimuths: [204.21052492]
  Time: 2460602.371204
  Height: 2858.63 km
  Azimuth: 153.13°, Altitude: 26.01°
  Sun Azimuth: 204.21°, Sun Altitude: -47.92°
  Parameters array: [2858.62569261  153.13380894   26.00629608  204.2

In [26]:
from lumos.geometry import Surface
from lumos.brdf.library import BINOMIAL, PHONG
import lumos.conversions
import lumos.brdf.library
import lumos.calculator
import lumos.plot
import numpy as np
import astropy.time
import astropy.coordinates
import astropy.units as u



def sun_altazi(time, ob_location):
    obs_time = astropy.time.Time(time, format='jd')  
    aa_frame = astropy.coordinates.AltAz(obstime=obs_time, location=ob_location)
    sun_altaz = astropy.coordinates.get_sun(obs_time).transform_to(aa_frame)
    
    sun_alt = np.array(sun_altaz.alt.degree)  
    sun_az = np.array(sun_altaz.az.degree)
    
    print("Sun Altitudes:", sun_alt)
    print("Sun Azimuths:", sun_az)
    
    # If it's a single-element array, just return the scalar values
    if sun_alt.size == 1:
        return sun_az.item(), sun_alt.item()
    else:
        return sun_az, sun_alt



def intensity(parameters, surfaces, earth_brdf = None, diffuse_sphere = False, filter = False):
    
    
    satellite_height=parameters[0]*1000 # make sure that parameters actually gives it in km and you have to convert to m
    azimuths=parameters[1]
    altitudes=parameters[2]

    sun_azimuth=parameters[3]
    sun_altitude=parameters[4]
    
    
    if diffuse_sphere:
        intensities = diffuse_sphere_model.get_intensity(
            0.65,
            satellite_height,
            altitudes,
            azimuths,
            sun_altitude,
            sun_azimuth)
    else:
        intensities = lumos.calculator.get_intensity_observer_frame( #go to lumos library and see what units it needs for each
            surfaces,
            satellite_height,
            altitudes,
            azimuths,
            sun_altitude,
            sun_azimuth,
            include_sun = True,
            include_earthshine = earth_brdf != None,
            earth_panel_density = 251,
            earth_brdf = earth_brdf
        )

    if filter:
        intensities = scipy.ndimage.gaussian_filter(intensities, 2, mode = ('wrap', 'reflect'))
        
    calculated_magnitudes = lumos.conversions.intensity_to_ab_mag(intensities)
        
    return calculated_magnitudes

    
    
# Define Earth location with height (assumed 0 m unless specified)
south_af = astropy.coordinates.EarthLocation(lat=-29.038169452952143*u.deg, lon=26.402995668269295*u.deg, height=1372*u.m)
# Julian Date example
observation_times = [2460602.371181]   
satellite_height=2849.99 #in Km
azimuths=153.26 #in degree
altitudes=26.18 #in degree

sun_azimuth,sun_altitude=sun_altazi(observation_times,south_af)


para=np.array([satellite_height,azimuths,altitudes,sun_azimuth,sun_altitude])



Sun Altitudes: [-47.91604432]
Sun Azimuths: [204.22193462]


In [27]:
#one_web_mmt9(overall)(data_points= +10,000)

oneweb_mmt_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 3.01))
]

oneweb_mmt_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.35, 1.56, 12.48)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 3.84))
]


#starlink 1.0 (krantz2023)(data_points=4724)

star1_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 4.62))
]

star1_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.72, 0.39, 16.49)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 1.58))
]



#starlink 1.0 Visorsat (krantz2023)(data_points=4707)

star1v_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.26, 0.86))
]

star1v_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.58)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.17, 0.28, 0.69))
]



#starlink 1.0 Visorsat (krantz2023)(data_points=4707)

star1d_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.26, 0.86))
]

star1d_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.58)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.17, 0.28, 0.69))
]

#starlink 1.5 (krantz2023)(data_points=2539)

star15_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.03, 2.81))
]

star15_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.35, 0.26, 7.50)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.16, 0.19, 0.57))
]


#Oneweb (krantz2023)(data_points=4216)

oneweb_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 6.23))
]

oneweb_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.15, 0.08, 6.35)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.08, 0.04, 8.78))
]


#Fankhauser_paper
#V1.5
#2D

# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to data measured by Scatterworks.
# The script for fitting is "lab_brdf_fits.ipynb"

B = np.array([[3.34, -98.085]])
C = np.array([[-999.999, 867.538, 1000., 1000., -731.248, 618.552, -294.054, 269.248, -144.853, 75.196]])
lab_chassis_brdf = BINOMIAL(B, C, d = 3.0, l1 = -5)

B = np.array([[0.534, -20.409]])
C = np.array([[-527.765, 1000., -676.579, 430.596, -175.806, 57.879]])
lab_solar_array_brdf = BINOMIAL(B, C, d = 3.0, l1 = -3)

#v1.5
v15_SURFACES_LAB_BRDFS_2D = [
    Surface(chassis_area, chassis_normal, lab_chassis_brdf),
    Surface(solar_array_area, solar_array_normal, lab_solar_array_brdf)
]

# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to actual satellite brightness observations.
# The script for fitting is "infer_brdfs.ipynb"

#v1.5
v15_SURFACES_INFER_BRDFS_2D = [
    Surface(1.0, chassis_normal, PHONG(0.34, 0.40, 8.9)),
    Surface(1.0, solar_array_normal, PHONG(0.15, 0.25, 0.26))]
    
    

#Fankhauser_paper

#6D
    
# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to data measured by Scatterworks.
# The script for fitting is "lab_brdf_fits.ipynb"

B = np.array([[3.34, -98.085]])
C = np.array([[-999.999, 867.538, 1000., 1000., -731.248, 618.552, -294.054, 269.248, -144.853, 75.196]])
lab_chassis_brdf = BINOMIAL(B, C, d = 3.0, l1 = -5)

B = np.array([[0.534, -20.409]])
C = np.array([[-527.765, 1000., -676.579, 430.596, -175.806, 57.879]])
lab_solar_array_brdf = BINOMIAL(B, C, d = 3.0, l1 = -3)

#v1.5
v15_SURFACES_LAB_BRDFS_6D = [
    Surface(chassis_area, chassis_normal, lab_chassis_brdf),
    Surface(chassis_area, np.array([0, 0, 1]), lab_chassis_brdf),
    Surface(0.2, np.array([0, -1, 0]), lab_chassis_brdf),
    Surface(0.2, np.array([0, 1, 0]), lab_chassis_brdf),
    Surface(0.1825, np.array([-1, 0, 0]), lab_chassis_brdf),
    Surface(0.1825, np.array([1, 0, 0]), lab_chassis_brdf),
    Surface(solar_array_area, solar_array_normal, lab_solar_array_brdf)
]
#v1.5
# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to actual satellite brightness observations.
# The script for fitting is "infer_brdfs.ipynb"
v15_SURFACES_INFER_BRDFS_6D = [
    Surface(1.0, chassis_normal, PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,0,1]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,1,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,-1,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([1,0,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([-1,0,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, solar_array_normal, PHONG(0, 0.14, 1.48))
]

#v2.0
surfaces2 = satellitemodels.get_surfaces()

Using interpolated chassis


In [20]:
import json
import numpy as np
from collections import Counter

# ------------------------------------------------------------
# 1) LOAD STARLINK GENERATION DATA + BUILD LOOKUP
# ------------------------------------------------------------
with open("starlink_generations.json", "r") as f:
    starlink_data = json.load(f)

# "STARLINK-1907" -> "v1", "v1.5", "v2 mini", etc.
starlink_version_lookup = {
    d["sat_name"]: d.get("generation", "v1")
    for d in starlink_data.get("satellites", [])
}

# Surfaces
surfaces_by_version = {
    "v1": {
        "6d": star1_surfaces6d,
        "2d": star1_surfaces2d,
    },
    "v1.5": {
        "6d": [star15_surfaces6d,v15_SURFACES_LAB_BRDFS_6D,v15_SURFACES_INFER_BRDFS_6D]
        "2d": star15_surfaces2d,v15_SURFACES_LAB_BRDFS_2D,v15_SURFACES_INFER_BRDFS_2D] # remove "6d" for these because there as many 6d and 2d surfaces so make sure all are available to select
    },
    "v2 mini": {
        "all": surfaces2, #treat srufaces2 the same as "star1_surfaces#, there is only one surface
    },
} #add oneweb

# Datasets
datasets = {
    "a": a,
    "b": b,
    "c": c,
    "d": d,
    "e": e,
    "F": F,
}

# Intensity Computation
results = []

for dataset_name, dataset in datasets.items():
    print(f"\nProcessing dataset '{dataset_name}' with {len(dataset)} rows...")

    for row in dataset:
        sat_full = row["satellite"]                 
        sat_name = sat_full.split(" (")[0].strip()  

        # Starlink only
        if "STARLINK" not in sat_name.upper():
            continue

        # Version from JSON (fallback to v1)
        version = starlink_version_lookup.get(sat_name, "v1")

        # Select surfaces (fallback to v1 if version not registered)
        surf_pack = surfaces_by_version.get(version, surfaces_by_version["v1"])

        if "all" in surf_pack:
            surfaces_6d = surf_pack["all"]
            surfaces_2d = None
        else:
            surfaces_6d = surf_pack["6d"]
            surfaces_2d = surf_pack["2d"]
        #change these two lines to select every surface necessary for v1.5 and v2.0

        # Single-point parameter array (5, 1)
        para = np.asarray(row["parameters"]).reshape(5, 1)

        # Compute intensities
        I6 = intensity(para, surfaces_6d)
        I2 = intensity(para, surfaces_2d) if surfaces_2d is not None else None

        # Store one result per point
        results.append({
            "dataset": dataset_name,
            "julian_date": row["julian_date"],
            "satellite": sat_name,
            "version": version,
            "I6D": I6,
            "I2D": I2,
        })

# Print all points
print("\n" + "="*90)
print("ALL STARLINK INTENSITY POINTS (ALL DATASETS)")
print("="*90)

for i, r in enumerate(results, start=1):
    print(
        f"{i:5d} | "
        f"set={r['dataset']} | "
        f"JD={r['julian_date']:.6f} | "
        f"{r['satellite']} | "
        f"ver={r['version']} | "
        f"I6D={r['I6D']} | "
        f"I2D={r['I2D']}"
    )
#dont print every point, find the mean and standard deviation of each of starlink satellites and print that

TypeError: unhashable type: 'list'

In [35]:
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# 1) SURFACE MODELS
# ------------------------------------------------------------
surfaces_by_version = {

    "v1": {
        "6d": {
            "star1":  star1_surfaces6d,
            "star1v": star1v_surfaces6d,
            "star1d": star1d_surfaces6d,
        },
        "2d": {
            "star1":  star1_surfaces2d,
            "star1v": star1v_surfaces2d,
            "star1d": star1d_surfaces2d,
        },
    },

    "v1.5": {
        "6d": {
            "krantz": star15_surfaces6d,
            "lab":    v15_SURFACES_LAB_BRDFS_6D,
            "infer":  v15_SURFACES_INFER_BRDFS_6D,
        },
        "2d": {
            "krantz": star15_surfaces2d,
            "lab":    v15_SURFACES_LAB_BRDFS_2D,
            "infer":  v15_SURFACES_INFER_BRDFS_2D,
        },
    },

    "v2 mini": {
        "6d": {
            "v2mini": surfaces2,
        },
        "2d": {
            "v2mini": surfaces2,
        },
    },

    "oneweb": {
        "6d": {
            "oneweb":     oneweb_surfaces6d,
            "oneweb_mmt": oneweb_mmt_surfaces6d,
        },
        "2d": {
            "oneweb":     oneweb_surfaces2d,
            "oneweb_mmt": oneweb_mmt_surfaces2d,
        },
    },
}

# ------------------------------------------------------------
# 2) DATASETS
# ------------------------------------------------------------
datasets = {"a": a, "b": b, "c": c, "d": d, "e": e, "F": F}

# ------------------------------------------------------------
# 3) COMPUTE INTENSITIES
# ------------------------------------------------------------
results = []

for dataset_name, dataset in datasets.items():
    for row in dataset:
        sat_name = row["satellite"].split(" (")[0].strip()
        sat_upper = sat_name.upper()

        # ✅ CORRECT VERSION LOGIC
        if "STARLINK" in sat_upper:
            version = starlink_version_lookup.get(sat_name, "v1")
        elif "ONEWEB" in sat_upper:
            version = "oneweb"
        else:
            continue

        surf_pack = surfaces_by_version[version]
        para = np.asarray(row["parameters"]).reshape(5, 1)

        I6 = {k: float(intensity(para, v)) for k, v in surf_pack["6d"].items()}
        I2 = {k: float(intensity(para, v)) for k, v in surf_pack["2d"].items()}

        results.append({
            "satellite": sat_name,
            "version": version,
            "I6D": I6,
            "I2D": I2,
        })

# ------------------------------------------------------------
# 4) SUMMARY PER SATELLITE PER MODEL
# ------------------------------------------------------------
summary = defaultdict(lambda: {
    "version": None,
    "I6D": defaultdict(list),
    "I2D": defaultdict(list),
})

for r in results:
    sat = r["satellite"]
    summary[sat]["version"] = r["version"]

    for m, v in r["I6D"].items():
        summary[sat]["I6D"][m].append(v)

    for m, v in r["I2D"].items():
        summary[sat]["I2D"][m].append(v)

# ------------------------------------------------------------
# 5) PRINT EVERYTHING (STARLINK + ONEWEB)
# ------------------------------------------------------------
print("\nINTENSITY SUMMARY (ALL SATELLITES, ALL MODELS)\n")

for sat in sorted(summary):
    data = summary[sat]
    print(f"\n{sat} | version = {data['version']}")

    for model, vals in data["I6D"].items():
        arr = np.array(vals)
        print(f"  6D [{model:10s}] mean={arr.mean():.4f} std={arr.std(ddof=1):.4f}")

    for model, vals in data["I2D"].items():
        arr = np.array(vals)
        print(f"  2D [{model:10s}] mean={arr.mean():.4f} std={arr.std(ddof=1):.4f}")



INTENSITY SUMMARY (ALL SATELLITES, ALL MODELS)


ONEWEB-0212 | version = oneweb
  6D [oneweb    ] mean=9.6563 std=0.1004
  6D [oneweb_mmt] mean=7.7569 std=0.0992
  2D [oneweb    ] mean=9.6800 std=0.1005
  2D [oneweb_mmt] mean=7.7026 std=0.0982

ONEWEB-0225 | version = oneweb
  6D [oneweb    ] mean=9.6714 std=0.0314
  6D [oneweb_mmt] mean=7.7505 std=0.0316
  2D [oneweb    ] mean=9.6956 std=0.0314
  2D [oneweb_mmt] mean=7.6772 std=0.0319

ONEWEB-0594 | version = oneweb
  6D [oneweb    ] mean=9.9230 std=0.0896
  6D [oneweb_mmt] mean=7.8011 std=0.1096
  2D [oneweb    ] mean=9.9456 std=0.0902
  2D [oneweb_mmt] mean=7.5255 std=0.1317

STARLINK-1169 | version = v1
  6D [star1     ] mean=6.9750 std=0.0120
  6D [star1v    ] mean=7.9916 std=0.0120
  6D [star1d    ] mean=7.9916 std=0.0120
  2D [star1     ] mean=6.7993 std=0.0124
  2D [star1v    ] mean=7.9794 std=0.0120
  2D [star1d    ] mean=7.9794 std=0.0120

STARLINK-1907 | version = v1
  6D [star1     ] mean=12.5000 std=0.0000
  6D [star1v   